# Pivot, Unpivot and Crosstabs

## Quick notes
### What this covers
- Turning rows into columns (pivot) and columns into rows (unpivot) in SQLite, which has no `PIVOT` keyword.
- Conditional aggregation with `CASE` and `FILTER`, counts vs sums, avoiding `NULL` holes.
- Crosstabs with a totals row and column, percent-of-total tables, ranking matrices, pivots with window sums.
- Dynamic columns (build the SQL text in Python), unpivot with `UNION ALL`, `json_each` and `VALUES`, transposing small tables.
### Key points
- Long format = one row per fact (`region, quarter, amount`). Wide format = one column per category value. Store long, report wide.
- Pivot recipe: `GROUP BY` the row key, one `SUM(CASE WHEN col = 'X' THEN amount END)` per new column.
- `SUM(...) FILTER (WHERE col = 'X')` says the same thing more readably.
- SQL needs the column list written in advance. If the values change, generate the SQL text in the host language.
- Unpivot recipe: `UNION ALL` one `SELECT` per column, or expand a small JSON object with `json_each`.
### Gotchas (the part worth revising twice)
- A `SUM` over zero matching rows is `NULL`, not 0. A `COUNT` over zero rows is 0. Wrap sums in `COALESCE(..., 0)`.
- A hard-coded pivot silently ignores any new category (a Q5 or a new product). Check a total against the unpivoted total.
- Forgetting the `ELSE` is fine for `SUM` (NULL is skipped) but `ELSE 0` is needed if you want zeros without `COALESCE`.
- Percent-of-total with integers: `100 * a / b` is integer division. Use `100.0`.
- Unpivoting keeps `NULL` cells as rows. Filter them out if "no value" should mean "no row".
### When it matters
- Management reports (quarters across the top), survey results, feature flags per user, exports to spreadsheets, and cleaning up wide imports before analysis.
### One-line recap example
```sql
SELECT region, SUM(amount) FILTER (WHERE quarter = 'Q1') AS q1, SUM(amount) FILTER (WHERE quarter = 'Q2') AS q2
FROM sales GROUP BY region;
```

## 01. Seed data
- `sales`: long format, several rows per region and quarter, with one deliberate hole (East has no Q3 sales) and a second product (Ink) that only appears in some cells.
- `survey`: a small wide table (one column per question) that we will unpivot later.
- Grouping and aggregation basics: see `01-Basics` and `02-Querying`. Window functions: `03-Window-Functions/01-Window-Functions`.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE sales (id INTEGER PRIMARY KEY, region TEXT, product TEXT, quarter TEXT, amount INTEGER);
INSERT INTO sales VALUES
 (1,'North','Pen','Q1',100),(2,'North','Pen','Q2',120),(3,'North','Ink','Q2',80),
 (4,'North','Pen','Q3',90),(5,'North','Ink','Q4',150),(6,'North','Pen','Q4',60),
 (7,'South','Pen','Q1',70),(8,'South','Ink','Q1',30),(9,'South','Pen','Q2',110),
 (10,'South','Pen','Q3',130),(11,'South','Pen','Q4',100),
 (12,'East','Pen','Q1',40),(13,'East','Ink','Q2',60),(14,'East','Pen','Q2',20),
 (15,'East','Pen','Q4',200),(16,'East','Ink','Q4',25);

CREATE TABLE survey (resp INTEGER PRIMARY KEY, q_price INTEGER, q_speed INTEGER, q_support INTEGER);
INSERT INTO survey VALUES (1,5,4,3),(2,4,NULL,2),(3,5,5,4);
SELECT region, quarter, product, amount FROM sales ORDER BY region, quarter, product;

region | quarter | product | amount
-------+---------+---------+-------
East   | Q1      | Pen     | 40    
East   | Q2      | Ink     | 60    
East   | Q2      | Pen     | 20    
East   | Q4      | Ink     | 25    
East   | Q4      | Pen     | 200   
North  | Q1      | Pen     | 100   
North  | Q2      | Ink     | 80    
North  | Q2      | Pen     | 120   
North  | Q3      | Pen     | 90    
North  | Q4      | Ink     | 150   
North  | Q4      | Pen     | 60    
South  | Q1      | Ink     | 30    
South  | Q1      | Pen     | 70    
South  | Q2      | Pen     | 110   
South  | Q3      | Pen     | 130   
South  | Q4      | Pen     | 100   
(16 rows)



## 02. Long vs wide
- Long (what we stored): one row per fact. Easy to insert into, easy to filter and group.
- Wide (what people read): one column per category. Easy to read, hard to extend (a new quarter means a new column).
- A first look at the long data aggregated per cell: a "pivot-ready" result still in long form.

In [3]:
%%sql
SELECT region, quarter, SUM(amount) AS total FROM sales GROUP BY region, quarter ORDER BY region, quarter;

region | quarter | total
-------+---------+------
East   | Q1      | 40   
East   | Q2      | 80   
East   | Q4      | 225  
North  | Q1      | 100  
North  | Q2      | 200  
North  | Q3      | 90   
North  | Q4      | 210  
South  | Q1      | 100  
South  | Q2      | 110  
South  | Q3      | 130  
South  | Q4      | 100  
(11 rows)



## 03. Pivot with conditional aggregation (CASE)
- One row per region. One column per quarter. Each column adds up only the rows of its quarter.
- `CASE WHEN quarter = 'Q1' THEN amount END` returns the amount for Q1 rows and `NULL` for the rest. `SUM` skips `NULL`s.
- This works in every database, so it is worth knowing by heart.
- Look at East Q3: no rows, so `SUM` gives `NULL` (a hole). We fix it in section 06.

In [4]:
%%sql
SELECT region,
       SUM(CASE WHEN quarter = 'Q1' THEN amount END) AS q1,
       SUM(CASE WHEN quarter = 'Q2' THEN amount END) AS q2,
       SUM(CASE WHEN quarter = 'Q3' THEN amount END) AS q3,
       SUM(CASE WHEN quarter = 'Q4' THEN amount END) AS q4
FROM sales GROUP BY region ORDER BY region;

region | q1  | q2  | q3   | q4 
-------+-----+-----+------+----
East   | 40  | 80  | NULL | 225
North  | 100 | 200 | 90   | 210
South  | 100 | 110 | 130  | 100
(3 rows)



## 04. The same pivot with FILTER
- `aggregate FILTER (WHERE condition)` is shorter and says what you mean. Available in SQLite 3.30+ (also PostgreSQL).
- It also works for `COUNT`, `AVG`, `MAX` and any other aggregate.

In [5]:
%%sql
SELECT region,
       SUM(amount) FILTER (WHERE quarter = 'Q1') AS q1,
       SUM(amount) FILTER (WHERE quarter = 'Q2') AS q2,
       SUM(amount) FILTER (WHERE quarter = 'Q3') AS q3,
       SUM(amount) FILTER (WHERE quarter = 'Q4') AS q4
FROM sales GROUP BY region ORDER BY region;

region | q1  | q2  | q3   | q4 
-------+-----+-----+------+----
East   | 40  | 80  | NULL | 225
North  | 100 | 200 | 90   | 210
South  | 100 | 110 | 130  | 100
(3 rows)



## 05. Pivot of counts vs sums
- Sums answer "how much", counts answer "how many sales".
- For counts, `COUNT(*) FILTER (WHERE ...)`, or `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`. Never `COUNT(CASE WHEN ... THEN 1 ELSE 0 END)`: `COUNT` counts non-NULL values, and 0 is non-NULL, so it counts every row.
- A count over no rows gives 0 (not `NULL`), so counts have no holes.

In [6]:
%%sql
SELECT region,
       COUNT(*) FILTER (WHERE quarter = 'Q1') AS q1_n,
       COUNT(*) FILTER (WHERE quarter = 'Q2') AS q2_n,
       COUNT(*) FILTER (WHERE quarter = 'Q3') AS q3_n,
       COUNT(*) FILTER (WHERE quarter = 'Q4') AS q4_n
FROM sales GROUP BY region ORDER BY region;

region | q1_n | q2_n | q3_n | q4_n
-------+------+------+------+-----
East   | 1    | 2    | 0    | 2   
North  | 1    | 2    | 1    | 2   
South  | 2    | 1    | 1    | 1   
(3 rows)



In [7]:
%%sql
-- Bad: ELSE 0 inside COUNT counts every row, so every column equals the region's row count
SELECT region,
       COUNT(CASE WHEN quarter = 'Q1' THEN 1 ELSE 0 END) AS q1_wrong,
       COUNT(CASE WHEN quarter = 'Q1' THEN 1 END)        AS q1_right
FROM sales GROUP BY region ORDER BY region;

region | q1_wrong | q1_right
-------+----------+---------
East   | 5        | 1       
North  | 6        | 1       
South  | 5        | 2       
(3 rows)



## 06. Avoiding NULL holes with COALESCE
- Empty cells come out as `NULL`. In a report you want 0.
- Wrap each aggregate: `COALESCE(SUM(...), 0)`. Do it around the `SUM`, not inside the `CASE`, otherwise you add 0s needlessly (still correct, just noisier).
- Also remember what `NULL` does in arithmetic: `NULL + 5` is `NULL`, so a row total built from holes would vanish. Fill first.

In [8]:
%%sql
SELECT region,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q1'), 0) AS q1,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) AS q2,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS q3,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q4'), 0) AS q4
FROM sales GROUP BY region ORDER BY region;

region | q1  | q2  | q3  | q4 
-------+-----+-----+-----+----
East   | 40  | 80  | 0   | 225
North  | 100 | 200 | 90  | 210
South  | 100 | 110 | 130 | 100
(3 rows)



In [9]:
%%sql
-- the NULL trap: adding unfilled sums
SELECT region,
       SUM(amount) FILTER (WHERE quarter = 'Q2') + SUM(amount) FILTER (WHERE quarter = 'Q3') AS q2_plus_q3_bad,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) + COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS q2_plus_q3_good
FROM sales GROUP BY region ORDER BY region;

region | q2_plus_q3_bad | q2_plus_q3_good
-------+----------------+----------------
East   | NULL           | 80             
North  | 290            | 290            
South  | 240            | 240            
(3 rows)



## 07. Crosstab with a totals column and a totals row
- Totals column: just `SUM(amount)` next to the pivoted columns.
- Totals row: SQLite has no `ROLLUP`, so `UNION ALL` the same query without the `GROUP BY`. Wrap the `UNION ALL` in a subquery so you can `ORDER BY region = 'TOTAL'` and keep the total row last (a compound select only allows plain column names in `ORDER BY`).
- The grand total in the corner must equal both the sum of the row totals and the sum of the four quarter totals (here 1385): a good sanity check.

In [10]:
%%sql
WITH p AS (
  SELECT region,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q1'), 0) AS q1,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) AS q2,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS q3,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q4'), 0) AS q4,
         SUM(amount) AS total
  FROM sales GROUP BY region)
SELECT * FROM (
  SELECT region, q1, q2, q3, q4, total FROM p
  UNION ALL
  SELECT 'TOTAL', SUM(q1), SUM(q2), SUM(q3), SUM(q4), SUM(total) FROM p)
ORDER BY region = 'TOTAL', region;

region | q1  | q2  | q3  | q4  | total
-------+-----+-----+-----+-----+------
East   | 40  | 80  | 0   | 225 | 345  
North  | 100 | 200 | 90  | 210 | 600  
South  | 100 | 110 | 130 | 100 | 440  
TOTAL  | 240 | 390 | 220 | 535 | 1385 
(4 rows)



## 08. Percent-of-total tables
- Percent of the grand total: each cell divided by the whole. Percent of the row: each cell divided by its row total (rows add to 100).
- Use `100.0 *` so the division is not integer division, and `ROUND` for display.
- Windows make it short: `SUM(...) OVER ()` for the grand total, `SUM(...) OVER (PARTITION BY region)` for the row total, applied to the long, per-cell data. Then pivot that.

In [11]:
%%sql
WITH cell AS (SELECT region, quarter, SUM(amount) AS amt FROM sales GROUP BY region, quarter),
pct AS (SELECT region, quarter, 100.0 * amt / SUM(amt) OVER (PARTITION BY region) AS p FROM cell)
SELECT region,
       ROUND(COALESCE(SUM(p) FILTER (WHERE quarter = 'Q1'), 0), 1) AS q1_pct,
       ROUND(COALESCE(SUM(p) FILTER (WHERE quarter = 'Q2'), 0), 1) AS q2_pct,
       ROUND(COALESCE(SUM(p) FILTER (WHERE quarter = 'Q3'), 0), 1) AS q3_pct,
       ROUND(COALESCE(SUM(p) FILTER (WHERE quarter = 'Q4'), 0), 1) AS q4_pct
FROM pct GROUP BY region ORDER BY region;

region | q1_pct | q2_pct | q3_pct | q4_pct
-------+--------+--------+--------+-------
East   | 11.6   | 23.2   | 0.0    | 65.2  
North  | 16.7   | 33.3   | 15.0   | 35.0  
South  | 22.7   | 25.0   | 29.5   | 22.7  
(3 rows)



In [12]:
%%sql
-- share of the grand total for each region (long form), plus the integer-division trap
SELECT region, SUM(amount) AS total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 1) AS pct_of_all,
       100 * SUM(amount) / SUM(SUM(amount)) OVER () AS pct_integer_division
FROM sales GROUP BY region ORDER BY region;

region | total | pct_of_all | pct_integer_division
-------+-------+------------+---------------------
East   | 345   | 24.9       | 24                  
North  | 600   | 43.3       | 43                  
South  | 440   | 31.8       | 31                  
(3 rows)



## 09. Pivot with window sums (running total across quarters)
- A window over the long data adds a running total per region. Pivot that to get "year to date" columns.
- Compute per-cell sums first (so each region/quarter is one row), then the window, then the pivot.
- Watch East: it has no Q3 row, so it has no running value for Q3 and `ytd_q3` shows 0 (not the 120 you would expect), while `ytd_q4` is right at 345. A missing period breaks a running total pivot. See the calendar trick in `04-Patterns/02-Gaps-Islands-And-Sessions` to fill missing periods properly.

In [13]:
%%sql
WITH cell AS (SELECT region, quarter, SUM(amount) AS amt FROM sales GROUP BY region, quarter),
ytd AS (SELECT region, quarter, SUM(amt) OVER (PARTITION BY region ORDER BY quarter) AS run FROM cell)
SELECT region,
       COALESCE(MAX(run) FILTER (WHERE quarter = 'Q1'), 0) AS ytd_q1,
       COALESCE(MAX(run) FILTER (WHERE quarter = 'Q2'), 0) AS ytd_q2,
       COALESCE(MAX(run) FILTER (WHERE quarter = 'Q3'), 0) AS ytd_q3,
       COALESCE(MAX(run) FILTER (WHERE quarter = 'Q4'), 0) AS ytd_q4
FROM ytd GROUP BY region ORDER BY region;

region | ytd_q1 | ytd_q2 | ytd_q3 | ytd_q4
-------+--------+--------+--------+-------
East   | 40     | 120    | 0      | 345   
North  | 100    | 300    | 390    | 600   
South  | 100    | 210    | 340    | 440   
(3 rows)



## 10. Ranking matrix
- Rank the regions inside each quarter, then pivot the ranks: rows are regions, columns are quarters, cells are ranks.
- Use `RANK` for ties (see `04-Patterns/01-Top-N-And-Ranking-Patterns`). A region with no sales in a quarter has no rank there (`NULL`) and is not ranked at all in that quarter.
- Pivot ranks with `MAX`/`MIN` (there is exactly one value per cell), not `SUM`, so that the meaning stays clear.

In [14]:
%%sql
WITH cell AS (SELECT region, quarter, SUM(amount) AS amt FROM sales GROUP BY region, quarter),
rk AS (SELECT region, quarter, RANK() OVER (PARTITION BY quarter ORDER BY amt DESC) AS r FROM cell)
SELECT region,
       MAX(r) FILTER (WHERE quarter = 'Q1') AS q1_rank,
       MAX(r) FILTER (WHERE quarter = 'Q2') AS q2_rank,
       MAX(r) FILTER (WHERE quarter = 'Q3') AS q3_rank,
       MAX(r) FILTER (WHERE quarter = 'Q4') AS q4_rank
FROM rk GROUP BY region ORDER BY region;

region | q1_rank | q2_rank | q3_rank | q4_rank
-------+---------+---------+---------+--------
East   | 3       | 3       | NULL    | 1      
North  | 1       | 1       | 2       | 2      
South  | 1       | 2       | 1       | 3      
(3 rows)



## 11. Pivoting text (MAX on strings)
- You can pivot non-numbers too. Use `MAX` or `MIN` (they work on text). Each cell then holds one string.
- Here: the best-selling product name per region and quarter. The `ROW_NUMBER` with a tie-breaker (`product`) makes the pick deterministic, so `MAX` only has one string per cell to return.
- First get one product per region and quarter (largest amount), then pivot it.

In [15]:
%%sql
WITH cell AS (SELECT region, quarter, product, SUM(amount) AS amt FROM sales GROUP BY region, quarter, product),
best AS (
  SELECT region, quarter, product,
         ROW_NUMBER() OVER (PARTITION BY region, quarter ORDER BY amt DESC, product) AS rn
  FROM cell)
SELECT region,
       MAX(product) FILTER (WHERE quarter = 'Q1') AS q1_best,
       MAX(product) FILTER (WHERE quarter = 'Q2') AS q2_best,
       MAX(product) FILTER (WHERE quarter = 'Q3') AS q3_best,
       MAX(product) FILTER (WHERE quarter = 'Q4') AS q4_best
FROM best WHERE rn = 1 GROUP BY region ORDER BY region;

region | q1_best | q2_best | q3_best | q4_best
-------+---------+---------+---------+--------
East   | Pen     | Ink     | NULL    | Pen    
North  | Pen     | Pen     | Pen     | Ink    
South  | Pen     | Pen     | Pen     | Pen    
(3 rows)



## 12. Two row dimensions: region and product down the side
- Add a second `GROUP BY` key (`product`) and you get a pivot with a compound row label.
- Pivots are just `GROUP BY` plus conditional columns, so every grouping idea still works.

In [16]:
%%sql
SELECT region, product,
       COALESCE(SUM(amount) FILTER (WHERE quarter IN ('Q1','Q2')), 0) AS h1,
       COALESCE(SUM(amount) FILTER (WHERE quarter IN ('Q3','Q4')), 0) AS h2
FROM sales GROUP BY region, product ORDER BY region, product;

region | product | h1  | h2 
-------+---------+-----+----
East   | Ink     | 60  | 25 
East   | Pen     | 60  | 200
North  | Ink     | 80  | 150
North  | Pen     | 220 | 150
South  | Ink     | 30  | 0  
South  | Pen     | 180 | 230
(6 rows)



## 13. Dynamic columns: generate the SQL in Python
- Plain SQL cannot decide the number of columns at run time. The column list must be in the statement text.
- So: ask for the distinct values first, build the pivot SQL string in the host language (here Python), then run it.
- Safety: category values go inside SQL text here, so quote them with a real escape (`value.replace("'", "''")`) or, better, only accept values you read from your own table. Never paste user input into SQL this way; pass it as bound parameters (`?` placeholders) instead.
- A new quarter such as `Q5` appears as a new column automatically on the next run.

In [17]:
quarters = [r[0] for r in con.execute("SELECT DISTINCT quarter FROM sales ORDER BY quarter")]
cols = ",\n  ".join(
    "COALESCE(SUM(amount) FILTER (WHERE quarter = '%s'), 0) AS \"%s\"" % (q.replace("'", "''"), q.replace('"', '""'))
    for q in quarters
)
pivot_sql = f"SELECT region,\n  {cols}\nFROM sales GROUP BY region ORDER BY region"
print(pivot_sql, "\n")
cur = con.execute(pivot_sql)
_table([d[0] for d in cur.description], cur.fetchall())

SELECT region,
  COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q1'), 0) AS "Q1",
  COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) AS "Q2",
  COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS "Q3",
  COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q4'), 0) AS "Q4"
FROM sales GROUP BY region ORDER BY region 

region | Q1  | Q2  | Q3  | Q4 
-------+-----+-----+-----+----
East   | 40  | 80  | 0   | 225
North  | 100 | 200 | 90  | 210
South  | 100 | 110 | 130 | 100
(3 rows)



## 14. The hard-coded pivot silently misses new values
- Add a Q5 row. The fixed Q1..Q4 pivot from section 07 ignores it: its row totals and the column cells no longer add up to the real total.
- Guard: compare the pivot's grand total to `SELECT SUM(amount)` from the raw table. Section 13's generated SQL would include Q5.
- We roll back the extra row afterwards.

In [18]:
%%sql
BEGIN;
INSERT INTO sales VALUES (17,'North','Pen','Q5',999);
SELECT (SELECT SUM(amount) FROM sales) AS real_total,
       (SELECT SUM(amount) FROM sales WHERE quarter IN ('Q1','Q2','Q3','Q4')) AS pivot_would_show,
       (SELECT SUM(amount) FROM sales) - (SELECT SUM(amount) FROM sales WHERE quarter IN ('Q1','Q2','Q3','Q4')) AS silently_missing;
ROLLBACK;
SELECT SUM(amount) AS total_after_rollback FROM sales;

real_total | pivot_would_show | silently_missing
-----------+------------------+-----------------
2384       | 1385             | 999             
(1 row)

total_after_rollback
--------------------
1385                
(1 row)



## 15. Unpivot with UNION ALL
- Reverse direction: wide to long. One `SELECT` per column, each labelling the value with the column's name.
- Works everywhere. Verbose when there are many columns.
- `NULL` answers (resp 2 skipped the "speed" question) come through as `NULL` rows. Filter `WHERE answer IS NOT NULL` if missing should mean absent.

In [19]:
%%sql
SELECT resp, 'price' AS question, q_price AS answer FROM survey
UNION ALL SELECT resp, 'speed',   q_speed   FROM survey
UNION ALL SELECT resp, 'support', q_support FROM survey
ORDER BY resp, question;

resp | question | answer
-----+----------+-------
1    | price    | 5     
1    | speed    | 4     
1    | support  | 3     
2    | price    | 4     
2    | speed    | NULL  
2    | support  | 2     
3    | price    | 5     
3    | speed    | 5     
3    | support  | 4     
(9 rows)



In [20]:
%%sql
-- aggregate per question: COUNT(answer) and AVG skip the NULL answers on their own
WITH long AS (
  SELECT resp, 'price' AS question, q_price AS answer FROM survey
  UNION ALL SELECT resp, 'speed',   q_speed   FROM survey
  UNION ALL SELECT resp, 'support', q_support FROM survey)
SELECT question, COUNT(answer) AS answers, ROUND(AVG(answer), 2) AS avg_score
FROM long GROUP BY question ORDER BY question;

question | answers | avg_score
---------+---------+----------
price    | 3       | 4.67     
speed    | 2       | 4.5      
support  | 3       | 3.0      
(3 rows)



## 16. Unpivot with json_each
- Build a JSON object from the columns, then `json_each` turns each key/value pair into a row. One expression instead of N `SELECT`s.
- The column names are still typed once, but the shape of the query does not grow.
- Values come back with their real type. A `NULL` value stays `NULL`.

In [21]:
%%sql
SELECT s.resp, j.key AS question, j.value AS answer
FROM survey s,
     json_each(json_object('price', s.q_price, 'speed', s.q_speed, 'support', s.q_support)) j
ORDER BY s.resp, j.key;

resp | question | answer
-----+----------+-------
1    | price    | 5     
1    | speed    | 4     
1    | support  | 3     
2    | price    | 4     
2    | speed    | NULL  
2    | support  | 2     
3    | price    | 5     
3    | speed    | 5     
3    | support  | 4     
(9 rows)



## 17. Unpivot with VALUES (cross join to a list)
- Cross join each row to a small `VALUES` list of (label, column index) pairs and pick the value with `CASE`.
- It is a "poor man's LATERAL" and keeps the labels in plain sight. SQLite cannot name the columns of an inline `VALUES` alias, so put it in a CTE with a column list.
- Handy when you want to add extra info per column (such as a weight or display order).

In [22]:
%%sql
WITH k(col, question, weight) AS (VALUES (1,'price',3),(2,'speed',2),(3,'support',1))
SELECT s.resp, k.question,
       CASE k.col WHEN 1 THEN s.q_price WHEN 2 THEN s.q_speed WHEN 3 THEN s.q_support END AS answer,
       k.weight
FROM survey s CROSS JOIN k
WHERE CASE k.col WHEN 1 THEN s.q_price WHEN 2 THEN s.q_speed WHEN 3 THEN s.q_support END IS NOT NULL
ORDER BY s.resp, k.col;

resp | question | answer | weight
-----+----------+--------+-------
1    | price    | 5      | 3     
1    | speed    | 4      | 2     
1    | support  | 3      | 1     
2    | price    | 4      | 3     
2    | support  | 2      | 1     
3    | price    | 5      | 3     
3    | speed    | 5      | 2     
3    | support  | 4      | 1     
(8 rows)



## 18. Transposing a small table
- Transpose = swap rows and columns. It is unpivot followed by pivot on the other axis.
- Here the survey (one row per respondent) becomes one row per question and one column per respondent.
- Only sensible for small, fixed-size tables, usually for display.

In [23]:
%%sql
WITH long AS (
  SELECT resp, 'price' AS question, q_price AS answer FROM survey
  UNION ALL SELECT resp, 'speed',   q_speed   FROM survey
  UNION ALL SELECT resp, 'support', q_support FROM survey)
SELECT question,
       MAX(answer) FILTER (WHERE resp = 1) AS resp_1,
       MAX(answer) FILTER (WHERE resp = 2) AS resp_2,
       MAX(answer) FILTER (WHERE resp = 3) AS resp_3
FROM long GROUP BY question ORDER BY question;

question | resp_1 | resp_2 | resp_3
---------+--------+--------+-------
price    | 5      | 4      | 5     
speed    | 4      | NULL   | 5     
support  | 3      | 2      | 4     
(3 rows)



## 19. Round trip: unpivot then re-pivot, and check
- A good test of a pivot: turn the wide result back into long and compare with the source totals.
- Here we pivot the sales by region, unpivot with `json_each`, and check that no money was lost by comparing sums.
- `json_each` on `json_object(...)` makes this short even for the pivot's own columns.

In [24]:
%%sql
WITH wide AS (
  SELECT region,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q1'), 0) AS q1,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) AS q2,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS q3,
         COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q4'), 0) AS q4
  FROM sales GROUP BY region),
back AS (
  SELECT w.region, j.key AS quarter, j.value AS amount
  FROM wide w, json_each(json_object('Q1', w.q1, 'Q2', w.q2, 'Q3', w.q3, 'Q4', w.q4)) j)
SELECT (SELECT SUM(amount) FROM back) AS after_round_trip,
       (SELECT SUM(amount) FROM sales) AS original,
       (SELECT COUNT(*) FROM back WHERE amount = 0) AS zero_filled_cells;

after_round_trip | original | zero_filled_cells
-----------------+----------+------------------
1385             | 1385     | 1                
(1 row)



## 20. Putting it together: a one-query management report
- Row per region, quarters across, row total, share of company total, and a rank by total. Each piece used earlier in this notebook.
- Windows over an aggregate (`SUM(SUM(amount)) OVER ()`, `RANK() OVER (ORDER BY SUM(amount) DESC)`) work because windows run after `GROUP BY`.

In [25]:
%%sql
SELECT region,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q1'), 0) AS q1,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q2'), 0) AS q2,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q3'), 0) AS q3,
       COALESCE(SUM(amount) FILTER (WHERE quarter = 'Q4'), 0) AS q4,
       SUM(amount) AS total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 1) AS pct_of_company,
       RANK() OVER (ORDER BY SUM(amount) DESC) AS place
FROM sales GROUP BY region ORDER BY place, region;

region | q1  | q2  | q3  | q4  | total | pct_of_company | place
-------+-----+-----+-----+-----+-------+----------------+------
North  | 100 | 200 | 90  | 210 | 600   | 43.3           | 1    
South  | 100 | 110 | 130 | 100 | 440   | 31.8           | 2    
East   | 40  | 80  | 0   | 225 | 345   | 24.9           | 3    
(3 rows)

